## Asteroid classification

The data is from [AstDys](https://newton.spacedys.com/astdys2/index.php?pc=5), datasets "Numbered and multiopposition asteroids" and "Individual asteroid family membership."

Load the data:

In [22]:
import pandas as pd
import numpy as np
family_membership_all = pd.read_csv("data/all_tro.members.txt", sep=r"\s+", comment="%", low_memory=False)
proper_elements = pd.read_csv("data/proper_elements.txt", sep=r"\s+", comment="%", low_memory=False)

Check that the data got imported correctly:

In [23]:
proper_elements.head()

,Name,mag,a,e,sinI,n,g,s,LCEx1E6,My
0,1,3.41,2.767096,0.116198,0.167585,78.193318,54.070272,-59.170034,1.16,-2
1,2,4.17,2.770918,0.281258,0.547547,78.041654,-1.335344,-46.393342,44.20,10
2,3,5.36,2.669366,0.233506,0.229226,82.528181,43.635655,-61.222138,13.74,-2
3,4,3.31,2.361513,0.098758,0.111336,99.188833,36.872897,-39.597863,2.83,2
4,6,5.76,2.425271,0.158486,0.247863,95.303184,31.568209,-41.829042,4.56,2


In [24]:
family_membership_all.head()

,name,Hmag,status,family1,dv_fam1,near1,family2,dv_fam2,near2,rescod
0,2,4.17,3,2,0.0,0,0,0.0,0,0
1,3,5.36,3,3,0.0,0,0,0.0,0,0
2,4,3.31,3,4,0.0,0,0,0.0,0,0
3,5,6.93,3,5,0.0,0,0,0.0,0,0
4,10,5.43,3,10,0.0,0,0,0.0,0,8-4-3


Split the family membership data into training, test, and validation sets. To do this, we'll get all the families that exist, then randomly split them into 3 groups, then cross-reference asteroid elements into training/test/validation sets to match.

In [31]:
families = pd.unique(family_membership_all["family1"])
rng = np.random.default_rng(seed=42)
rng.shuffle(families)
[families_train_raw, families_test_raw, families_validate_raw] = np.array_split(families, 3)
# convert from numpy to dataframe
families_train = pd.DataFrame(families_train_raw, columns=["family1"])
families_test = pd.DataFrame(families_test_raw, columns=["family1"])
families_validate = pd.DataFrame(families_validate_raw, columns=["family1"])
# reconstruct family_membership_[train|test|validate]
family_membership_train = family_membership_all[family_membership_all["family1"].isin(families_train["family1"])]
family_membership_test = family_membership_all[family_membership_all["family1"].isin(families_test["family1"])]
family_membership_validate = family_membership_all[family_membership_all["family1"].isin(families_validate["family1"])]

In [32]:
proper_elements_train = proper_elements[proper_elements["Name"].isin(family_membership_train["name"])]
proper_elements_test = proper_elements[proper_elements["Name"].isin(family_membership_test["name"])]
proper_elements_validate = proper_elements[proper_elements["Name"].isin(family_membership_validate["name"])]
proper_elements_no_family = proper_elements[~proper_elements["Name"].isin(family_membership_all["name"])].sample(frac=1, random_state=rng)
# split up the no-family asteroids into train, test, and validate as well so there's some representation of them in each set
[proper_elements_no_family_train_raw, proper_elements_no_family_test_raw, proper_elements_no_family_validate_raw] = np.array_split(proper_elements_no_family, 3)
proper_elements_no_family_train = pd.DataFrame(proper_elements_no_family_train_raw, columns=proper_elements.columns)
proper_elements_no_family_test = pd.DataFrame(proper_elements_no_family_test_raw, columns=proper_elements.columns)
proper_elements_no_family_validate = pd.DataFrame(proper_elements_no_family_validate_raw, columns=proper_elements.columns)
proper_elements_train = pd.concat([proper_elements_train, proper_elements_no_family_train])
proper_elements_test = pd.concat([proper_elements_test, proper_elements_no_family_test])
proper_elements_validate = pd.concat([proper_elements_validate, proper_elements_no_family_validate])

### Validation

In [39]:
# check that the proper element sets are disjoint
assert set(proper_elements_train["Name"]).isdisjoint(set(proper_elements_test["Name"]))
assert set(proper_elements_train["Name"]).isdisjoint(set(proper_elements_validate["Name"]))
assert set(proper_elements_test["Name"]).isdisjoint(set(proper_elements_validate["Name"]))
# check that the proper element sets sum to the original set
assert len(proper_elements_train) + len(proper_elements_test) + len(proper_elements_validate) == len(proper_elements)
# check that the proper elements include both family and no-family asteroids
assert len(set(proper_elements_train["Name"]).intersection(set(family_membership_train["name"]))) > 0, "proper_elements_train should include some family asteroids"
assert len(set(proper_elements_train["Name"]).intersection(set(family_membership_train["name"]))) < len(proper_elements_train), "proper_elements_train should include some no-family asteroids"
assert len(set(proper_elements_test["Name"]).intersection(set(family_membership_test["name"]))) > 0, "proper_elements_test should include some family asteroids"
assert len(set(proper_elements_test["Name"]).intersection(set(family_membership_test["name"]))) < len(proper_elements_test), "proper_elements_test should include some no-family asteroids"
assert len(set(proper_elements_validate["Name"]).intersection(set(family_membership_validate["name"]))) > 0, "proper_elements_validate should include some family asteroids"
assert len(set(proper_elements_validate["Name"]).intersection(set(family_membership_validate["name"]))) < len(proper_elements_validate), "proper_elements_validate should include some no-family asteroids"
# check that the family membership sets are disjoint
assert set(family_membership_train["name"]).isdisjoint(set(family_membership_test["name"]))
assert set(family_membership_train["name"]).isdisjoint(set(family_membership_validate["name"]))
assert set(family_membership_test["name"]).isdisjoint(set(family_membership_validate["name"]))
# check that the family membership sets sum to the original set
assert len(family_membership_train) + len(family_membership_test) + len(family_membership_validate) == len(family_membership_all)
# check that no families are split across sets
assert set(family_membership_train["family1"]).isdisjoint(set(family_membership_test["family1"])), "a family is split across train and test"
assert set(family_membership_train["family1"]).isdisjoint(set(family_membership_validate["family1"])), "a family is split across train and validate"
assert set(family_membership_test["family1"]).isdisjoint(set(family_membership_validate["family1"])), "a family is split across test and validate"
print("All checks passed!")
family_membership_train.to_csv("data/family_membership_train.csv", index=False)
family_membership_test.to_csv("data/family_membership_test.csv", index=False)
family_membership_validate.to_csv("data/family_membership_validate.csv", index=False)
proper_elements_train.to_csv("data/proper_elements_train.csv", index=False)
proper_elements_test.to_csv("data/proper_elements_test.csv", index=False)
proper_elements_validate.to_csv("data/proper_elements_validate.csv", index=False)


All checks passed!
